## 智能问答功能完整设计

---

### 一、整体架构（RAG + SSE 流式）

```
用户提问 → 前端 → SSE 流式接口 → 后端
                                    ↓
                            ┌─ 会话管理（创建/获取）
                            ├─ 历史消息（取最近10条）
                            ├─ RAG 检索
                            │   ├─ retriever 向量召回（top_k=10）
                            │   └─ reranker 重排序（top_k=3）
                            ├─ 构造 System Prompt（含参考资料）
                            ├─ LLM 流式生成
                            └─ SSE 流式返回（session → context → token → done）
                                    ↓
                            前端实时渲染 → 保存 AI 回复
```

---

### 二、涉及文件清单

表格

| 层级 | 文件路径 | 作用 |
| --- | --- | --- |
| **前端页面** | `frontend/src/pages/ChatPage.jsx` | 主页面，处理输入、流式接收、渲染 |
| **前端组件** | `frontend/src/components/ChatInput.jsx` | 输入框组件 |
| **前端组件** | `frontend/src/components/MemoMessageBubble.jsx` | 消息气泡（Markdown 渲染） |
| **后端路由** | `backend/app/api/chat.py` | 聊天接口，会话管理、RAG 流程 |
| **后端服务** | `backend/app/services/llm.py` | LLM 调用（chat、chat_stream） |
| **后端服务** | `backend/app/services/retriever.py` | 向量检索（召回知识库） |
| **后端服务** | `backend/app/services/reranker.py` | 重排序（精排） |
| **数据模型** | `backend/app/models/document.py` | ChatSession、ChatMessage、Document |

---

### 三、前端关键流程

#### 1. 发起请求（handleAsk）

```
await fetchEventSource(`${API_BASE}/api/chat/stream`, {
  method: 'POST',
  body: JSON.stringify({ question: q, session_id: sessionId }),
  onmessage(ev) {
    const data = JSON.parse(ev.data);
    if (data.type === 'token') {
      answer += data.content;  // 累加 token
      scheduleRender();         // requestAnimationFrame 节流渲染
    }
    else if (data.type === 'context') sources = data.contexts;  // 参考资料
    else if (data.type === 'session') setSessionId(data.session_id);  // 新会话
  },
  onclose() { setLoading(false); }
});
```

#### 2. 核心技术点

- ✅ **SSE 流式**：用 `@microsoft/fetch-event-source` 处理 Server-Sent Events
- ✅ **节流渲染**：`requestAnimationFrame` 减少重绘次数
- ✅ **超时保护**：60 秒无响应自动断开
- ✅ **历史对话目录**：右侧边栏展示所有问题，点击跳转
- ✅ **Markdown 渲染**：消息气泡用 Markdown 渲染代码块、表格

---

### 四、后端关键流程

#### 1. 主接口 `/api/chat/stream`

```
@router.post("/stream")
def chat_stream_api(req: ChatRequest, db: Session = Depends(get_db)):
    # 1. 创建/获取会话
    if req.session_id:
        session_id = req.session_id
    else:
        # 用 LLM 生成 10 字内标题
        title = chat([...])  # 标题生成
        session = ChatSession(title=title)
        db.add(session); db.commit()
        session_id = str(session.id)

    # 2. 存用户消息
    db.add(ChatMessage(session_id=session_id, role="user", content=req.question))

    # 3. 取最近 10 条历史
    history = db.query(ChatMessage).filter(...).limit(10).all()

    # 4. RAG 检索
    candidates = retrieve(req.question, top_k=10)  # 向量召回
    reranked = rerank(req.question, docs, top_k=3)   # 重排序

    # 5. 构造 System Prompt
    system_prompt = f"""你是「AI 学习工作台」的智能助手...
    参考资料：{context_text}"""

    # 6. SSE 流式返回
    def event_generator():
        yield session      # 会话 ID
        yield context     # 参考资料
        for token in chat_stream(messages):
            yield token   # 流式 token
        yield done       # 完成
```

#### 2. RAG 核心流程

```
用户问题
    ↓
retrieve（向量召回，top_k=10）  →  从知识库向量库召回 10 个相关片段
    ↓
rerank（重排序，top_k=3）       →  用 CrossEncoder 精排，取前 3 个
    ↓
构造 Prompt                     →  把 3 个片段塞进 System Prompt
    ↓
chat_stream（流式生成）         →  LLM 逐 token 输出
```

---

### 五、数据模型

#### ChatSession（会话表）

```
class ChatSession(Base):
    id: UUID
    title: str          # 会话标题
    pinned: bool        # 是否置顶
    is_deleted: bool    # 软删除
    created_at: datetime
```

#### ChatMessage（消息表）

```
class ChatMessage(Base):
    id: UUID
    session_id: UUID    # 关联会话
    role: str           # user / assistant
    content: str        # 消息内容
    feedback: str       # 点赞/点踩
    is_deleted: bool
    created_at: datetime
```

---

### 六、特色功能

表格

| 功能 | 实现方式 |
| --- | --- |
| **流式输出** | SSE + fetch-event-source + requestAnimationFrame 节流 |
| **历史对话目录** | 右侧边栏展示所有用户问题，点击平滑滚动到对应位置 |
| **引用标注** | 参考资料用 `[[1]]` 标注在正文后 |
| **代码格式严格** | System Prompt 强制要求代码换行、缩进 |
| **会话管理** | 自动生成标题、置顶、删除、分享 |
| **限流** | 1 分钟最多 10 次请求 |
| **会话锁** | 防止同一会话并发请求 |

---

这就是智能问答的完整设计！

## 问题1：对比国内外开源agent，在做输入意图识别时是怎么做的？

### 主流大模型产品的设计

#### 1. DeepSeek / 豆包 / Kimi / GPT / Claude 的共性设计

```
用户问题
    ↓
┌─────────────────────────────────────┐
│ 1. 意图分类（Router / Classifier）  │  ← 轻量模型或规则
│   - 闲聊 / 问答 / 工具调用 / 创作    │
└─────────────────────────────────────┘
    ↓
┌─────────────────────────────────────┐
│ 2. Query 理解（改写 / 扩展）        │  ← LLM
│   - 指代消解 / 上下文补全 / 关键词提取 │
└─────────────────────────────────────┘
    ↓
┌─────────────────────────────────────┐
│ 3. 路由分发                         │
│   - RAG 知识库检索                  │
│   - 工具调用（搜索/计算器/代码执行） │
│   - 直接生成                        │
└─────────────────────────────────────┘
    ↓
┌─────────────────────────────────────┐
│ 4. 生成回答                         │
└─────────────────────────────────────┘
```

---

### 各产品具体实现

#### 1. GPT / Claude（OpenAI / Anthropic）

**设计思路：Function Calling + 多步推理**

```
用户问题
    ↓
LLM 直接判断：
    - 需不需要调用工具？
    - 调用哪个工具？
    - 工具参数是什么？
    ↓
工具调用 → 拿到结果 → LLM 继续推理 → 最终回答
```

**特点**：

- ✅ 意图识别内置在 LLM 里（Tool Use）
- ✅ 支持多轮工具调用（ReAct 模式）
- ✅ 不需要额外的分类器模型
- ✅ 靠 Prompt 引导模型选择工具

---

#### 2. DeepSeek（深度求索）

**设计思路：轻量 Router + RAG**

```
用户问题
    ↓
轻量分类模型（7B 或更小）：
    - 闲聊 / 问答 / 工具调用
    ↓
路由分发：
    - 闲聊 → 直接回复
    - 问答 → RAG 检索 + 生成
    - 工具调用 → Function Calling
```

**特点**：

- ✅ 专门的 Router 模型做意图分类
- ✅ 成本低（轻量模型）
- ✅ 准确率高（专门训练过）

---

#### 3. 豆包（字节跳动）

**设计思路：多轮对话管理 + 意图槽位填充**

```
用户问题
    ↓
对话管理器（DM）：
    - 维护对话状态
    - 识别当前意图
    - 填充槽位（比如：订机票需要出发地、目的地、时间）
    ↓
槽位填齐 → 调用对应技能
槽位缺失 → 反问用户补充信息
```

**特点**：

- ✅ 面向任务型对话（订机票、查酒店等）
- ✅ 槽位填充（Slot Filling）
- ✅ 多轮对话状态跟踪

---

#### 4. Kimi（月之暗面）

**设计思路：长文本理解 + 自动检索判断**

```
用户问题
    ↓
LLM 判断：
    - 这个问题需要检索资料吗？
    - 还是直接回答？
    ↓
如果需要检索 → RAG
如果不需要 → 直接生成
```

**特点**：

- ✅ 意图识别内置在 LLM 里
- ✅ 长上下文窗口（128K+）
- ✅ 自动判断是否需要检索

---

### 对比总结

表格

| 维度 | 当前项目 | GPT/Claude | DeepSeek | 豆包 | Kimi |
| --- | --- | --- | --- | --- | --- |
| **意图识别方式** | 硬编码字符串 | Function Calling | 专门 Router 模型 | 对话管理器 | LLM 内置判断 |
| **模型大小** | 无（纯规则） | 大模型本身 | 轻量 7B | 专门 DM | 大模型本身 |
| **支持意图** | 3 种 | 无限（自定义工具） | 多种 | 任务型对话 | 检索 / 不检索 |
| **多轮对话** | 无 | 有（ReAct） | 有 | 强（槽位填充） | 有 |
| **成本** | 极低 | 高 | 中 | 高 | 高 |
| **准确率** | 低 | 高 | 高 | 很高 | 高 |